<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Módulo 4: Feature Hashing</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

Feature Hashing (o *hashing trick*) transforma categorías en un número fijo de columnas mediante una función hash. Es útil cuando el conjunto de categorías es muy grande o cambia con el tiempo: no necesita guardar un diccionario de todas las categorías y mantiene el mismo tamaño de salida.

La **cardinalidad** es el número de valores distintos de una variable. Una cardinalidad alta puede hacer que *one-hot encoding* genere una matriz con muchas columnas, la mayoría de ellas en cero. Hashing controla ese tamaño, pero acepta dos costos importantes: distintas categorías pueden colisionar en el mismo bucket y las columnas resultantes no son directamente interpretables.

El tamaño de salida se define con `n_features` (el número de *buckets*). Más buckets suelen reducir las colisiones, pero aumentan la memoria. No hay un umbral universal para decidir cuándo la cardinalidad es alta: evalúa el número de categorías, el tamaño del conjunto y los recursos disponibles.

En los ejemplos compararemos la idea de asignar categorías a buckets con las implementaciones de `FeatureHasher` y `category_encoders`. Observa que la salida es una representación, no un conteo de cuántas veces ocurrió cada categoría.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Data/Países.csv')
df.head()

In [ ]:
df.info()

In [ ]:
len(df)

In [ ]:
df['Países'].nunique(), df['Países'].unique()

In [ ]:
df['Países'].value_counts(normalize=False)

In [ ]:
vc=df.Países.value_counts()
vc.plot.bar(figsize=(15,5),grid=True)

In [ ]:
# Cardinalidad de 'Países'


In [ ]:
# índice 19, 20 categorías

In [ ]:
# 20 categorías generan el 81.6% de las observaciones

In [ ]:
# 19 categorías generan el 78.9% de las observaciones

### ¿Cómo funciona Feature hashing?

1. **Definir el número de buckets (m):**
   Se elige un número fijo de columnas (buckets) donde se mapearán las categorías. Este número es mucho menor que la cantidad total de categorías posibles.

2. **Aplicar una función hash:**
   Cada categoría se transforma usando una función hash (por ejemplo, hash de Python, MurmurHash, etc.), que convierte el valor de la categoría en un número entero.

3. **Asignar a un bucket:**
   El valor hash se reduce al rango de buckets usando la operación módulo (`hash(categoría) % m`). Así, cada categoría se asigna a uno de los m buckets.

4. **Codificación:**
   Para cada observación, se crea un vector de longitud m, donde solo el bucket correspondiente a la categoría tiene un valor distinto de cero (por ejemplo, 1 o la frecuencia), y el resto son ceros. Si varias categorías caen en el mismo bucket, sus valores se suman (esto se llama colisión).

### Algunas Ventajas:
- Permite trabajar con variables categóricas de alta cardinalidad sin crear miles de columnas.
- El espacio de representación es fijo y controlable.
- Es eficiente en memoria y velocidad.

### Riegos que puede haber con es:
- Puede haber colisiones: diferentes categorías pueden terminar en el mismo bucket, perdiendo información.
- No es interpretable: no es posible saber a qué categoría corresponde cada bucket.


#### Ejemplo

In [ ]:
vc

In [ ]:
n

In [ ]:
from sklearn.utils.murmurhash import murmurhash3_32

# Número fijo de buckets (columnas de salida)


#### Función hash

Una función hash asigna cada categoría a un entero; al tomar el módulo `m`, obtenemos un índice entre `0` y `m - 1`. Como hay más categorías posibles que buckets, dos o más categorías pueden compartir índice: eso es una **colisión**.

En esta demostración usamos MurmurHash3 para asignar de forma reproducible cada país a un bucket. El valor del bucket identifica una columna; no es el conteo de esa categoría. Más adelante, la clase `FeatureHasher` hará esta transformación y devolverá una matriz dispersa.

In [ ]:
# El conteo indica cuántas observaciones terminaron en cada bucket


In [ ]:
# Identifica los buckets a los que se asignaron varias categorías


In [ ]:
# Buckets con más de una observación; puede ser repetición o colisión


In [ ]:
# Número de buckets ocupados por al menos una categoría


In [ ]:
# Comparar categorías distintas que comparten el mismo bucket


### Función hash simple (basado en el módulo)

In [ ]:
# Mayor número de observaciones asignadas a un mismo bucket


In [ ]:
def hash_simple(values, n_features):
    """Asigna cada categoría a un bucket determinista entre 0 y n_features - 1."""
    return None

### Usando librerías

#### `FeatureHasher` de scikit-learn

`FeatureHasher` transforma cada observación en un vector disperso con un número fijo de columnas. Recibe una secuencia de diccionarios: las **claves** son nombres de características y los **valores** son números. Para representar una categoría, construimos una clave como `"color=rojo"` con valor `1`; no pasamos la etiqueta de texto como valor.

- `n_features` define el número de buckets de salida.
- `input_type='dict'` indica que cada observación es un diccionario de características numéricas.
- `alternate_sign=True` (predeterminado) puede asignar valores `-1` o `1`, lo que ayuda a reducir el sesgo de las colisiones.

La salida es una matriz dispersa. Convertirla con `.toarray()` facilita verla en un ejemplo pequeño, pero puede consumir mucha memoria en conjuntos grandes. El hashing no conserva un diccionario que permita recuperar directamente la categoría original.

In [ ]:
from sklearn.feature_extraction import FeatureHasher

data = [
    {'category=A': 1},
    {'category=B': 1},
    {'category=A': 1},
    {'category=C': 1}
]
# FeatureHasher


In [ ]:
#Otro ejemplo
data = [
    {'color=red': 1, 'shape=circle': 1},
    {'color=blue': 1, 'shape=square': 1},
    {'color=green': 1, 'shape=triangle': 1},
    {'color=red': 1, 'shape=square': 1}
]
data_df = pd.DataFrame(data).fillna(0)
data_df

In [ ]:
# Un bucket por característica; los choques se suman en la misma columna.

# La matriz dispersa almacena principalmente los valores distintos de cero.


**MurmurHash3** es una función hash no criptográfica, rápida y eficiente, ampliamente utilizada para aplicaciones como hash tables y feature hashing. La versión de 32 bits produce un entero de 32 bits a partir de una cadena o secuencia de bytes.

- **No criptográfica:** No es segura para aplicaciones de seguridad, pero es muy rápida y tiene baja tasa de colisiones para uso general.
- **Determinística:** La misma entrada y semilla siempre producen el mismo hash.
- **Uniformidad:** Distribuye los valores de hash de manera uniforme, lo que es ideal para hashing en estructuras de datos y machine learning.

En scikit-learn y otras librerías, MurmurHash3 se usa internamente para convertir cadenas en índices de buckets de manera eficiente y reproducible.


In [ ]:
from sklearn.utils.murmurhash import murmurhash3_32

# Ejemplo: obtener el hash de una cadena


In [ ]:
#help(murmurhash3_32)

In [ ]:
# Función hash MurmurHash3 de 32 bits
from sklearn.feature_extraction import FeatureHasher

In [ ]:
# Transformar cada país en una característica binaria y aplicar hashing


In [ ]:
# Cada fila representa una observación y cada columna un bucket


In [ ]:
# Localizar la columna no nula de cada fila para ilustrar el bucket asignado


Los índices calculados manualmente con MurmurHash3 y los de `FeatureHasher` no tienen que coincidir: los métodos pueden hashear cadenas distintas y aplicar signos diferentes. Lo que se mantiene es el tamaño fijo de salida y la posibilidad de colisiones; no uses los índices para recuperar la categoría original.

In [ ]:
#!pip install category-encoders

#### Category Encoders

`category_encoders` es un paquete adicional, independiente de scikit-learn. Si no está instalado en el entorno, ejecuta `%pip install category-encoders` en una celda antes de continuar. El ejemplo aplica hashing directamente a la columna `Países` y devuelve las columnas codificadas.

In [ ]:
#Usando category_encoders para dataset de países
from category_encoders import HashingEncoder

In [ ]:
#help(HashingEncoder)

In [ ]:
#help(fcnHash_OH.fit_transform)

Para organizar las ideas vistas hasta aquí:

1. **Diagnosticar cardinalidad:** revisa cuántas categorías hay y cómo se distribuyen sus frecuencias. Una variable con muchas categorías raras puede hacer que one-hot genere una matriz muy ancha y dispersa.
2. **Entender el hashing trick:** una función determinista asigna cada categoría a uno de un número fijo de buckets. No se necesita aprender ni guardar un vocabulario para transformar datos nuevos.
3. **Interpretar colisiones:** dos categorías distintas pueden terminar en el mismo bucket y sus señales se mezclan. Aumentar `n_features` suele reducir colisiones, pero aumenta el tamaño de la representación.
4. **Elegir una implementación:** `FeatureHasher` de scikit-learn trabaja directamente con una cantidad fija de columnas dispersas. `HashingEncoder` pertenece al paquete adicional `category_encoders`; sus parámetros y comportamiento deben revisarse por separado.
5. **Evaluar el costo de interpretabilidad:** los buckets no conservan el nombre de las categorías. Hashing resulta útil cuando escala y memoria importan más que explicar el efecto de cada categoría.

El ejemplo con códigos de país ayuda a inspeccionar los buckets. El ejercicio siguiente amplía la práctica a un dataset real de alta cardinalidad y compara hashing con one-hot bajo la misma evaluación.

## Ejercicio aplicado: hashing de direcciones en ventas de Melbourne

El archivo `Data/melb_data.csv` contiene ventas reales de propiedades de Melbourne. Usaremos `Address` para predecir `Price` con una regresión Ridge. Las direcciones tienen alta cardinalidad, por lo que one-hot puede producir muchas columnas. Compararemos esa representación con Feature Hashing de tamaño fijo.

El propósito es estudiar el comportamiento de la codificación, no construir el mejor modelo de precios. El mismo modelo y la misma partición se usarán para ambas alternativas; los resultados dependen de la partición y del número de buckets elegido.

### 1. Cargar y separar los datos

La partición se realiza antes de ajustar one-hot. FeatureHasher no necesita ajuste de vocabulario, pero separamos los datos antes de evaluar para comparar justamente y medir las colisiones entre las direcciones distintas del conjunto de entrenamiento.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import FeatureHasher
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

melbourne = pd.read_csv('Data/melb_data.csv', usecols=['Address', 'Price'])
melbourne = melbourne.dropna(subset=['Address', 'Price']).reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(
    melbourne[['Address']], melbourne['Price'],
    test_size=0.25, random_state=42
)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print(f'Observaciones: {len(melbourne):,}')
print(f'Direcciones distintas: {melbourne["Address"].nunique():,}')
print(f'Filas de entrenamiento: {len(X_train):,} | prueba: {len(X_test):,}')
melbourne.head()

### 2. Crear las representaciones

Cada dirección se entrega a FeatureHasher como una característica categórica binaria, con el prefijo `Address=`. `alternate_sign=False` hace que los valores no nulos sean positivos, lo cual facilita su interpretación en este ejercicio. La matriz hash se mantiene dispersa y siempre tiene `n_features` columnas.

One-hot aprende las direcciones de entrenamiento y omite las desconocidas de prueba (`handle_unknown='ignore'`). Para hashing, una dirección nueva también se asigna a un bucket, aunque puede colisionar con otra dirección.

In [ ]:
# One-hot: una columna por cada dirección observada en entrenamiento.

# Hashing: tamaño fijo, sin guardar un vocabulario de direcciones.


### 3. Evaluar y revisar colisiones

Entrenamos el mismo modelo con ambas representaciones y medimos MAE, RMSE y R² en el mismo conjunto de prueba. Luego calculamos cuántos buckets reciben direcciones distintas en entrenamiento. Las repeticiones de una misma dirección no cuentan como colisión; una colisión ocurre cuando direcciones distintas comparten bucket.

In [ ]:
# Hash de cada dirección distinta para contar buckets ocupados y colisiones.


### 4. Ventajas, desventajas y aplicaciones reales

| Codificación | Ventajas | Desventajas |
|---|---|---|
| **Feature Hashing** | Número fijo de columnas; no requiere mantener vocabulario; transforma categorías nuevas sin reentrenar el codificador; puede ser eficiente en datos dispersos de alta cardinalidad. | Colisiones mezclan categorías; las columnas no son interpretables; el resultado depende del número de buckets y de la función/hash de nombres elegida. |
| **One-hot** | Fácil de interpretar; cada categoría conocida tiene su propia columna y coeficiente. | La cantidad de columnas crece con el vocabulario; necesita aprender y mantener categorías; categorías nuevas requieren una política explícita. |
| **Conteo o frecuencia** | Una columna compacta y sencilla, sin utilizar el objetivo. | Categorías distintas pueden compartir valor y la frecuencia puede no aportar señal predictiva. |
| **Codificación por media** | Compacta e incorpora la relación entre categoría y variable objetivo. | Usa el objetivo y puede sobreajustar o filtrar información; requiere validación out-of-fold y suavizado apropiados. |

**Lectura de este ejercicio:** compara las métricas y el número de columnas. One-hot conserva mejor la identidad, pero puede ser muy ancho; hashing limita las columnas a `n_buckets`, a cambio de aceptar colisiones. Un resultado predictivo no demuestra que hashing sea siempre mejor o peor: depende de cardinalidad, datos, modelo y dimensionalidad elegida.

**Casos reales donde puede ser apropiado:**

- texto y n-gramas en clasificación de correo no deseado, temas o sentimientos, donde el vocabulario puede crecer continuamente;
- identificadores de URL, productos, consultas o dispositivos en sistemas de recomendación y publicidad digital;
- registros de eventos y telemetría con nombres de eventos, códigos o etiquetas de alta cardinalidad;
- variables categóricas de alto volumen en modelos en línea o distribuidos, cuando no es práctico coordinar un vocabulario completo.

En cada caso se debe medir el impacto de colisiones y elegir `n_features` con validación. Feature Hashing no es una función hash criptográfica y no debe usarse para proteger datos sensibles.
